this notebook aims to verify whether there are any missing interractions between groups not withing the same main groups. this willw user to have a better idea on whether the predictions would be reliable or not.

In [2]:
# import packages
import sys
import os
import pickle
sys.path.append(os.path.join(os.getcwd(), "../"))
import pandas as pd
from utils.gc import load_gc, inspect_interractions, inspect_contribs
import warnings


# load group contributions
df_gc, df_a = load_gc('../data/data_unifac.xlsx', 'dataframe')
df_a

,CH3,CH2,CH,C,CH2=CH,CH=CH,CH2=C,CH=C,C=C,ACH,...,CH2OCH,CH2OC,CHOCH,CHOC,COC,O=COC=O,AC-CN,AC-Br,CH2SuCH2,CH2SuCH
CH3,0.00,0.00,0.00,0.00,86.02,86.02,86.02,86.02,86.02,61.13,...,808.59,808.59,808.59,808.59,808.59,408.3,718.01,0.0,0.0,0.0
CH2,0.00,0.00,0.00,0.00,86.02,86.02,86.02,86.02,86.02,61.13,...,808.59,808.59,808.59,808.59,808.59,408.3,718.01,0.0,0.0,0.0
CH,0.00,0.00,0.00,0.00,86.02,86.02,86.02,86.02,86.02,61.13,...,808.59,808.59,808.59,808.59,808.59,408.3,718.01,0.0,0.0,0.0
C,0.00,0.00,0.00,0.00,86.02,86.02,86.02,86.02,86.02,61.13,...,808.59,808.59,808.59,808.59,808.59,408.3,718.01,0.0,0.0,0.0
CH2=CH,-35.36,-35.36,-35.36,-35.36,0.00,0.00,0.00,0.00,0.00,38.81,...,200.94,200.94,200.94,200.94,200.94,219.9,-677.25,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O=COC=O,21.49,21.49,21.49,21.49,-2.80,-2.80,-2.80,-2.80,-2.80,344.42,...,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
AC-CN,272.82,272.82,272.82,272.82,569.71,569.71,569.71,569.71,569.71,165.18,...,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
AC-Br,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,920.49,...,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
CH2SuCH2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0


let's unpack groups and make them into a dict and save this for the future

In [3]:
# read the sheet with the various main groups and subgroups
df = pd.read_excel('../data/data_unifac.xlsx', sheet_name='set')
# index based on subgroup idex
df.set_index('sub_g', inplace=True)
df.index.name = None
# create a dict
unifac_dict = df.to_dict(orient='index')
# Save dictionary to a file
with open('../data/unifac_groups.pkl', 'wb') as f:
    pickle.dump(unifac_dict, f)

unifac_dict
# we can use this to get groups of seletced main group
#main_g_C = next(value['main_g'] for value in unifac_dict.values() if value['grp'] == df_test.index[df_test.index == 'CH'].values[0])
#main_g_C

{1: {'main_g': 1, 'grp': 'CH3'},
 2: {'main_g': 1, 'grp': 'CH2'},
 3: {'main_g': 1, 'grp': 'CH'},
 4: {'main_g': 1, 'grp': 'C'},
 5: {'main_g': 2, 'grp': 'CH2=CH'},
 6: {'main_g': 2, 'grp': 'CH=CH'},
 7: {'main_g': 2, 'grp': 'CH2=C'},
 8: {'main_g': 2, 'grp': 'CH=C'},
 9: {'main_g': 2, 'grp': 'C=C'},
 10: {'main_g': 3, 'grp': 'ACH'},
 11: {'main_g': 3, 'grp': 'AC'},
 12: {'main_g': 4, 'grp': 'ACCH3'},
 13: {'main_g': 4, 'grp': 'ACCH2'},
 14: {'main_g': 4, 'grp': 'ACCH'},
 15: {'main_g': 5, 'grp': 'OH'},
 16: {'main_g': 6, 'grp': 'CH3OH'},
 17: {'main_g': 7, 'grp': 'H2O'},
 18: {'main_g': 8, 'grp': 'ACOH'},
 19: {'main_g': 9, 'grp': 'CH3CO'},
 20: {'main_g': 9, 'grp': 'CH2CO'},
 21: {'main_g': 10, 'grp': 'HCO'},
 22: {'main_g': 11, 'grp': 'CH3COO'},
 23: {'main_g': 11, 'grp': 'CH2COO'},
 24: {'main_g': 12, 'grp': 'HCOO'},
 25: {'main_g': 13, 'grp': 'CH3O'},
 26: {'main_g': 13, 'grp': 'CH2O'},
 27: {'main_g': 13, 'grp': 'CHO'},
 28: {'main_g': 13, 'grp': 'THF'},
 29: {'main_g': 14, 'grp'

we can check all the groups partaining to the same main group

we load the reduced interraction table with only main groups

We will construct a function that inspect the interractins of the groups chosen and provide a warning in case these are 0 and of different main groups.

In [4]:
grp_lst = ['CH3', 'CH3CO', 'CH3OH', 'H2O', 'DMSO', 'DOH', 'CH2', 'OH', 'CH', 'CH2CO', 'CHNH2']
df_test = df_a.loc[grp_lst,grp_lst]
df_test

,CH3,CH3CO,CH3OH,H2O,DMSO,DOH,CH2,OH,CH,CH2CO,CHNH2
CH3,0.00,476.40,697.2,1318.00,526.50,3025.00,0.00,986.50,0.00,476.40,391.50
CH3CO,26.76,0.00,108.7,472.50,-44.58,71.46,26.76,164.50,26.76,0.00,0.00
CH3OH,16.51,23.39,0.0,-181.00,-399.30,-119.20,16.51,249.10,16.51,23.39,359.30
H2O,300.00,-195.40,289.6,0.00,-139.00,12.72,300.00,-229.10,300.00,-195.40,48.89
DMSO,50.49,110.40,695.0,-240.00,0.00,535.80,50.49,-25.87,50.49,110.40,-366.51
DOH,140.00,135.40,240.8,-137.40,-417.20,0.00,140.00,267.60,140.00,135.40,0.00
CH2,0.00,476.40,697.2,1318.00,526.50,3025.00,0.00,986.50,0.00,476.40,391.50
OH,156.40,84.00,-137.1,353.50,-202.10,-318.90,156.40,0.00,156.40,84.00,83.02
CH,0.00,476.40,697.2,1318.00,526.50,3025.00,0.00,986.50,0.00,476.40,391.50
CH2CO,26.76,0.00,108.7,472.50,-44.58,71.46,26.76,164.50,26.76,0.00,0.00


In [5]:
# we inspect the interraction fo the unifac
inspect_interractions(df_test)

c:\Users\arnaou\PycharmProjects\camd_iss\scripts\..\utils\gc.py:96: UserWarning: 
Warning, A total of 3 out of 55 interaction parameters are missing
  warnings.warn(f"\nWarning, A total of {len(zero_locations)} out of {int_tot} interaction parameters are missing", UserWarning)
c:\Users\arnaou\PycharmProjects\camd_iss\scripts\..\utils\gc.py:98: UserWarning: 
Warning, the interaction between the following main groups [(9, 14), (31, 14), (9, 14)] have been set to 0
  warnings.warn(f"\nWarning, the interaction between the following main groups {missing_main_groups} have been set to 0", UserWarning)


([(9, 14), (31, 14), (9, 14)],
 [('CH3CO', 'CHNH2'), ('DOH', 'CHNH2'), ('CH2CO', 'CHNH2')])

In [6]:
# We inspect the cotributions of the seletced properties and groups
grp_lst = ['CH3', 'CH3CO','CH2']
inspect_contribs(df_gc.loc[grp_lst, :])

c:\Users\arnaou\PycharmProjects\camd_iss\scripts\..\utils\gc.py:36: UserWarning: Warning: some groups have a 0 contribution for the property: Column 'FPX' 
  warnings.warn(f"Warning: some groups have a 0 contribution for the property: Column '{col}' ")
c:\Users\arnaou\PycharmProjects\camd_iss\scripts\..\utils\gc.py:36: UserWarning: Warning: some groups have a 0 contribution for the property: Column 'EHX' 
  warnings.warn(f"Warning: some groups have a 0 contribution for the property: Column '{col}' ")
c:\Users\arnaou\PycharmProjects\camd_iss\scripts\..\utils\gc.py:36: UserWarning: Warning: some groups have a 0 contribution for the property: Column 'Gtt' 
  warnings.warn(f"Warning: some groups have a 0 contribution for the property: Column '{col}' ")


{'ccsub': ['CH3', 'CH3CO', 'CH2'],
 'FPX': ['CH3', 'CH2'],
 'EHX': ['CH3', 'CH2'],
 'Gtt': ['CH3CO']}